# Saliency and Observer Disagreement Prediction with a Center-Biased Deep Encoder-Decoder

Reproduces every number reported in the report: the main
comparison (Tab. 1), the ablation study (Tab. 2), the observer-disagreement
prediction results (Tab. 3), and the narrative statistics in the
"Observer Disagreement" section.

All logic lives in `src/`; this notebook only orchestrates.

## Toggles

| Variable | Effect |
|---|---|
| `RUN_TRAINING` | Train the base models and the agreement heads. |
| `RUN_EVALUATION` | Evaluate on MIT1003 and assemble the result tables. |
| `RUN_FULL_SWEEP` | `True`: 3 seeds per report-relevant tier (9 runs total). `False`: seed 0 only, which is all the report's tables need. |
| `USE_SAVED_CHECKPOINTS` | `True`: skip training/recomputation for anything already on disk. `False`: retrain and recompute everything. |
| `SHOW_PREDICTIONS` | `True`: plot a few MIT1003 predictions vs. ground truth for each evaluated variant. |
| `EVALUATE_ONE_MODEL` | `True`: evaluate just `ONE_MODEL_LABEL` on MIT1003, standalone from the rest of the notebook (see "Optional: evaluate a single model"). |

Run both `RUN_TRAINING` and `RUN_EVALUATION` as `True` for a full
reproduction from scratch (with `USE_SAVED_CHECKPOINTS = False`), or just
`RUN_EVALUATION` to rebuild the tables from checkpoints already present.


In [ ]:
RUN_TRAINING = True
RUN_EVALUATION = True

RUN_FULL_SWEEP = False        # True: 3 seeds x 3 tiers. False: seed 0 only (matches the report).
USE_SAVED_CHECKPOINTS = True  # True: skip training/recomputation where a checkpoint or cache file already exists.
SHOW_PREDICTIONS = True       # True: after evaluating each variant, plot a few MIT1003 predictions vs ground truth.

EVALUATE_ONE_MODEL = False    # True: evaluate just ONE_MODEL_LABEL on MIT1003, standalone (see "Optional: evaluate a single model").
ONE_MODEL_LABEL = "Ours_Fusion"
# ONE_MODEL_LABEL accepts one of REPORT_VARIANTS' keys:
#   "Ours_Res34"  - base_resnet,         resnet34 backbone, kldiv_cc loss (Tab. 1/2)
#   "Ours_Fusion" - intermediate_fusion, resnet34 backbone, kldiv_cc loss (the proposed model, Tab. 1/2)
#   "Ours_PVT"    - advanced_transformer, pvt_v2_b0 backbone, kldiv_cc loss (Tab. 2 ablation)
#   "Ours_Res18"  - base_resnet,         resnet18 backbone, kldiv_cc loss (Tab. 2 ablation)
#   "Ours_MSE"    - base_resnet,         resnet34 backbone, mse loss (Tab. 2 ablation)


## Setup

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Fill in before running on Colab.
GITHUB_OWNER = "nardib"
GITHUB_REPO = "saliency_prediction_neural_network"

if IN_COLAB:
    repo_dir = Path("/content") / GITHUB_REPO
    if not repo_dir.exists():
        # Public repo, no token needed. checkpoints/ and cache/ are not tracked
        # here (checkpoints exceed GitHub's 100MB file limit) — the notebook
        # trains and recomputes everything itself; upload a checkpoints/ or
        # cache/ folder into repo_dir manually first if you want to skip that.
        os.system(f"git clone --depth 1 https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git {repo_dir}")
    os.chdir(repo_dir)

assert Path("src").is_dir(), f"Cannot find src/ from {os.getcwd()}."
sys.path.insert(0, str(Path.cwd()))
print("working directory:", os.getcwd())


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.3", "gdown==5.2.0"],
               check=False)

## Environment

In [ ]:
import torch
from src.utils import load_config, set_seed, get_device
from src.setup_check import check_all

config = load_config("configs/config.yaml")
# configs/config.yaml is shared with the local notebook. Colab caps usable
# dataloader workers around 2; fall back to a higher count when this notebook
# is run outside Colab (e.g. testing it locally).
config["data"]["num_workers"] = 2 if IN_COLAB else 8
set_seed(config["seed"])
# No config passed here: the data hasn't been downloaded yet at this point in the
# notebook, so a data check now would always print FAIL. verify_dataset/verify_observers
# in the next cell are the real data sanity check, run right after the download.
check_all()

device = get_device()
print("device:", device)
if device.type == "cuda":
    print("gpu   :", torch.cuda.get_device_name(0))
    print("memory:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Data

Downloads SALICON, MIT1003 (aggregate maps), and MIT1003's per-observer
fixation records. Each call is a no-op if the data is already present.


In [ ]:
from src.download import download_salicon, download_mit1003, verify_dataset
from src.download_observers import download_mit1003_observers, verify_observers

salicon_root = download_salicon(file_id=config["data"]["salicon_id"],
                                local_root=config["data"]["salicon_root"])
mit1003_root = download_mit1003(config["data"]["mit1003_root"])
download_mit1003_observers(config["data"]["mit1003_root"])

verify_dataset(salicon_root, "train")
verify_dataset(salicon_root, "val")
print()
verify_observers(config["data"]["mit1003_root"])


## Model variants

Five configurations cover every row of Tab. 1 and Tab. 2. The three tier
variants (`Ours_Res34`, `Ours_Fusion`, `Ours_PVT`) also supply the base
checkpoints the agreement heads train on top of.


In [ ]:
from copy import deepcopy
from pathlib import Path

from src.utils import build_run_name

REPORT_VARIANTS = {
    "Ours_Res34":  dict(tier="base_resnet",         backbone="resnet34",   loss="kldiv_cc"),
    "Ours_Fusion": dict(tier="intermediate_fusion",  backbone="resnet34",   loss="kldiv_cc"),
    "Ours_PVT":    dict(tier="advanced_transformer", backbone="pvt_v2_b0",  loss="kldiv_cc"),
    "Ours_Res18":  dict(tier="base_resnet",         backbone="resnet18",   loss="kldiv_cc"),
    "Ours_MSE":    dict(tier="base_resnet",         backbone="resnet34",   loss="mse"),
}
# Only these three have a role in the agreement study (Tab. 3); the other two
# are ablation-only and are never trained beyond seed 0.
AGREEMENT_TIERS = {"Ours_Res34": "base_resnet", "Ours_Fusion": "intermediate_fusion", "Ours_PVT": "advanced_transformer"}
OUT_INDICES = {"intermediate_fusion": [2, 3, 4], "advanced_transformer": [1, 2, 3]}


def variant_config(spec: dict, seed: int = 0) -> dict:
    cfg = deepcopy(config)
    cfg["seed"] = seed
    cfg["model"]["name"] = spec["tier"]
    cfg["model"]["backbone"] = spec["backbone"]
    if spec["tier"] in OUT_INDICES:
        cfg["model"]["out_indices"] = OUT_INDICES[spec["tier"]]
    cfg["train"]["loss"] = spec["loss"]
    cfg["train"]["checkpoint_dir"] = str(
        Path(config["train"]["checkpoint_dir"]) / "sweep" / f"{spec['tier']}_{spec['backbone']}_{spec['loss']}_seed{seed}")
    cfg["train"]["resume_from"] = None
    return cfg


def variant_checkpoint_path(label: str, seed: int = 0) -> Path:
    cfg = variant_config(REPORT_VARIANTS[label], seed)
    return Path(cfg["train"]["checkpoint_dir"]) / f"{build_run_name(cfg)}_best.pt"


seeds_to_run = (0, 1, 2) if RUN_FULL_SWEEP else (0,)
# checkpoints[label] = {seed: path}, populated for every seed each variant needs
checkpoints = {
    label: {seed: variant_checkpoint_path(label, seed)
            for seed in (seeds_to_run if label in AGREEMENT_TIERS else (0,))}
    for label in REPORT_VARIANTS
}


### Optional: evaluate a single model

Standalone from the rest of the notebook — evaluates just one variant's
checkpoint on MIT1003 without needing the other four, and without feeding
the "Result tables" cell below (which expects all five). Useful for
checking a checkpoint you trained separately, or for a quick look without
running the full pipeline. Controlled by `EVALUATE_ONE_MODEL` and
`ONE_MODEL_LABEL` in the Toggles cell at the top.


In [ ]:
if EVALUATE_ONE_MODEL:
    from src.model import build_model
    from src.utils import load_checkpoint
    from src.evaluate import evaluate_mit1003

    cfg = variant_config(REPORT_VARIANTS[ONE_MODEL_LABEL], seed=0)
    ckpt_path = checkpoints[ONE_MODEL_LABEL][0]
    if not ckpt_path.exists():
        raise RuntimeError(
            f"{ckpt_path} not found. Train it first, or set ckpt_path to your own checkpoint's path.")

    model = build_model(cfg).to(device)
    load_checkpoint(str(ckpt_path), model, device=device)
    metrics = evaluate_mit1003(model, cfg, device)
    print(f"{ONE_MODEL_LABEL}: {metrics}")
else:
    print("EVALUATE_ONE_MODEL is False, skipping.")


## Training

### Train the base models

Skips any (variant, seed) whose checkpoint already exists when
`USE_SAVED_CHECKPOINTS` is `True`.


In [ ]:
if RUN_TRAINING:
    import matplotlib.pyplot as plt

    from src.data import get_dataloaders
    from src.model import build_model
    from src.train import train
    from src.visualize import plot_training_curves

    train_loader, val_loader = get_dataloaders(config)

    for label, spec in REPORT_VARIANTS.items():
        for seed, ckpt_path in checkpoints[label].items():
            if USE_SAVED_CHECKPOINTS and ckpt_path.exists():
                print(f"[skip] {label} seed {seed}: found {ckpt_path}")
                continue
            print("\n" + "=" * 70)
            print(f"TRAINING {label} (seed {seed})")
            print("=" * 70)
            cfg = variant_config(spec, seed)
            set_seed(seed)
            model = build_model(cfg)
            model, history = train(model, train_loader, val_loader, cfg)
            plot_training_curves(history)
            plt.show()
            del model
else:
    print("RUN_TRAINING is False, skipping.")


### Observer-agreement targets

Independent of model training. Computes the per-image Jensen-Shannon
excess divergence (`js_excess`, the agreement-head training target) from
MIT1003's per-observer fixation records, and reports the two data-quality
statistics quoted in the "Observer Disagreement" section of the report.


In [ ]:
if RUN_TRAINING:
    import numpy as np
    from src.observers import load_observers, inspect
    from src.agreement_targets import compute_dataset, report

    targets_path = Path(config["agreement"]["targets_path"])
    if USE_SAVED_CHECKPOINTS and targets_path.exists():
        print(f"[skip] agreement targets: found {targets_path}")
        targets = dict(np.load(targets_path))
    else:
        records = load_observers("mit1003", config["data"]["mit1003_root"], limit=50)
        inspect(records)
        records = load_observers("mit1003", config["data"]["mit1003_root"])
        print(f"{len(records)} images loaded")
        targets = compute_dataset(records, n_observers=15, seed=config["seed"])
        targets_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(targets_path, **targets)
        print("wrote", targets_path)

    gates = report(targets)

    finite = np.isfinite(targets["js_excess"])
    js_observed_mean = float(targets["js_observed"][finite].mean())
    js_null_mean = float(targets["js_null"][finite].mean())
    js_excess_mean = float(targets["js_excess"][finite].mean())
    print(f"\njs_observed : {js_observed_mean:.3f} bits")
    print(f"js_null     : {js_null_mean:.3f} bits ({js_null_mean / js_observed_mean:.0%} of observed)")
    print(f"js_excess   : {js_excess_mean:.3f} bits, positive for {gates['frac_positive']:.1%} of images")
    print(f"split-half reliability : {gates['ceiling']:.3f}")
    print(f"attainable ceiling (sqrt): {gates['ceiling'] ** 0.5:.3f}")
else:
    print("RUN_TRAINING is False, skipping.")


### Agreement heads

Cross-validates the agreement head on top of each of the three tiers that
appear in Tab. 3, using each tier's seed-0 checkpoint (head A stays frozen,
so the density metrics from the previous section are untouched). Also
produces the three-feature Ridge baseline and the split-half noise ceiling.


In [ ]:
if RUN_TRAINING:
    import json
    from src.train_agreement import run_cross_validation

    agreement_results = {}
    for label, tier in AGREEMENT_TIERS.items():
        out_path = Path("results") / f"agreement_{tier}.json"
        if USE_SAVED_CHECKPOINTS and out_path.exists():
            agreement_results[label] = json.loads(out_path.read_text())
            print(f"[skip] {label}: loaded cached result from {out_path}")
            continue

        print("\n" + "=" * 70)
        print(f"AGREEMENT HEAD on {label} ({tier})")
        print("=" * 70)
        spec = REPORT_VARIANTS[label]
        cfg = variant_config(spec, seed=0)
        cfg["agreement"]["base_checkpoint"] = str(checkpoints[label][0])
        cfg["agreement"]["freeze_base"] = True
        cfg["agreement"]["checkpoint_dir"] = str(Path(config["agreement"]["checkpoint_dir"]) / tier)
        agreement_results[label] = run_cross_validation(cfg, device=device, out_path=str(out_path))
else:
    print("RUN_TRAINING is False, skipping.")


## Evaluation

Evaluates every trained checkpoint on MIT1003 and assembles the three
result tables from the report.


In [ ]:
if RUN_EVALUATION:
    import json
    from src.evaluate import evaluate_mit1003
    from src.model import build_model
    from src.utils import load_checkpoint

    eval_path = Path("results/mit1003_eval.json")
    eval_results = json.loads(eval_path.read_text()) if eval_path.exists() else {}

    for label, spec in REPORT_VARIANTS.items():
        if USE_SAVED_CHECKPOINTS and label in eval_results:
            print(f"[skip] {label}: already in {eval_path}")
            continue
        ckpt_path = checkpoints[label][0]
        if not ckpt_path.exists():
            raise RuntimeError(
                f"{ckpt_path} not found. Run with RUN_TRAINING=True first, or set "
                f"USE_SAVED_CHECKPOINTS=False to retrain.")
        cfg = variant_config(spec, seed=0)
        model = build_model(cfg).to(device)
        load_checkpoint(str(ckpt_path), model, device=device)
        eval_results[label] = evaluate_mit1003(model, cfg, device)
        del model
        print(f"{label}: {eval_results[label]}")

    eval_path.parent.mkdir(exist_ok=True)
    with open(eval_path, "w") as f:
        json.dump(eval_results, f, indent=2)
else:
    print("RUN_EVALUATION is False, skipping.")


### Qualitative check: predicted saliency vs. ground truth

For each variant, on both test sets — SALICON validation and MIT1003 —
a few samples: original image, ground-truth fixation map, predicted map,
and the prediction overlaid on the image.

SALICON validation metrics printed here are a supplementary sanity check;
the report only reports MIT1003 numbers (Tab. 1/2), which is
what the table cell above already covers.


In [ ]:
if RUN_EVALUATION and SHOW_PREDICTIONS:
    import matplotlib.pyplot as plt
    from torch.utils.data import DataLoader

    from src.data import SaliencyDataset, get_mit1003_dataset
    from src.evaluate import evaluate
    from src.losses import SaliencyLoss
    from src.model import build_model
    from src.utils import load_checkpoint
    from src.visualize import print_metrics_table, plot_final_metrics, show_predictions

    salicon_val_dataset = SaliencyDataset(
        root=config["data"]["salicon_root"], split="val", img_size=config["data"]["img_size"],
        preserve_aspect_ratio=config["data"].get("preserve_aspect_ratio", True), return_mask=True,
    )
    salicon_val_loader = DataLoader(salicon_val_dataset, batch_size=config["data"]["batch_size"], shuffle=False)
    mit1003_dataset = get_mit1003_dataset(config)

    for label, spec in REPORT_VARIANTS.items():
        ckpt_path = checkpoints[label][0]
        if not ckpt_path.exists():
            print(f"[skip] {label}: {ckpt_path} not found.")
            continue

        cfg = variant_config(spec, seed=0)
        model = build_model(cfg).to(device)
        load_checkpoint(str(ckpt_path), model, device=device)

        criterion = SaliencyLoss(loss_type=cfg["train"]["loss"],
                                 kldiv_weight=cfg["train"].get("kldiv_weight", 1.0),
                                 cc_weight=cfg["train"].get("cc_weight", 0.5))
        salicon_metrics = evaluate(model, salicon_val_loader, criterion, device,
                                   aux_weight=cfg["train"].get("aux_weight", 1.0))
        print_metrics_table(salicon_metrics, title=f"{label} — SALICON validation")
        plot_final_metrics(salicon_metrics)
        plt.show()
        fig = show_predictions(model, salicon_val_dataset, device, n_samples=3)
        fig.suptitle(f"{label} — SALICON validation")
        plt.show()

        if mit1003_dataset is not None:
            if label in eval_results:
                print_metrics_table(eval_results[label], title=f"{label} — MIT1003")
                plot_final_metrics(eval_results[label])
                plt.show()
            fig = show_predictions(model, mit1003_dataset, device, n_samples=3)
            fig.suptitle(f"{label} — MIT1003")
            plt.show()

        del model


In [ ]:
if RUN_EVALUATION:
    import json
    import numpy as np

    agreement_results = {}
    for label, tier in AGREEMENT_TIERS.items():
        out_path = Path("results") / f"agreement_{tier}.json"
        if not out_path.exists():
            raise RuntimeError(
                f"{out_path} not found. Run with RUN_TRAINING=True first, or set "
                f"USE_SAVED_CHECKPOINTS=False to retrain.")
        agreement_results[label] = json.loads(out_path.read_text())

    targets_path = Path(config["agreement"]["targets_path"])
    if not targets_path.exists():
        raise RuntimeError(f"{targets_path} not found. Run with RUN_TRAINING=True first.")
    targets = dict(np.load(targets_path))
else:
    print("RUN_EVALUATION is False, skipping.")


## Result tables

Formatted to match the report.


In [ ]:
if RUN_EVALUATION:
    import pandas as pd
    from src.agreement_targets import report as report_gates

    pd.set_option("display.precision", 3)

    # --- Tab. 1: comparison with the literature (Ours_Fusion is the proposed model) ---
    literature = {
        "DVA":             {"cc": 0.64, "sim": 0.50, "kldiv": None},
        "MSI-Net":         {"cc": 0.79, "sim": 0.68, "kldiv": 0.66},
        "TranSalNet_Res":  {"cc": 0.76, "sim": 0.61, "kldiv": 0.78},
        "Ours_Fusion":     {k: eval_results["Ours_Fusion"][k] for k in ("cc", "sim", "kldiv")},
    }
    tab1 = pd.DataFrame(literature).T
    print("Tab. 1 — Comparison with other approaches on MIT1003")
    display(tab1)

    # --- Tab. 2: ablation study ---
    tab2 = pd.DataFrame({label: {k: eval_results[label][k] for k in ("cc", "sim", "kldiv", "nss", "auc_judd")}
                         for label in REPORT_VARIANTS}).T
    tab2 = tab2.loc[["Ours_Res34", "Ours_Res18", "Ours_PVT", "Ours_MSE", "Ours_Fusion"]]
    print("\nTab. 2 — Ablation study on MIT1003")
    display(tab2)

    # --- Tab. 3: observer-disagreement prediction ---
    gates = report_gates(targets)
    rows = {}
    for label in AGREEMENT_TIERS:
        res = agreement_results[label]
        rows[label] = {
            "per-fold rho": float(np.mean(res["per_fold_spearman"])),
            "pooled rho": res["pooled_spearman"],
        }
    tab3 = pd.DataFrame(rows).T
    tab3.loc["Three-feature baseline"] = {"per-fold rho": float("nan"),
                                          "pooled rho": agreement_results["Ours_Res34"]["baseline_spearman"]}
    tab3.loc["Attainable maximum"] = {"per-fold rho": float("nan"), "pooled rho": gates["ceiling"] ** 0.5}
    print("\nTab. 3 — Prediction of observer disagreement on MIT1003")
    display(tab3)
else:
    print("RUN_EVALUATION is False, skipping.")


---
**Not reproduced here:** the multi-seed tier-tie significance check, phase-0
diagnostics on a prior run, and the optional stage-3/log-density extras from
the exploratory workflow — none of them appear as a number in the report.
Set `RUN_FULL_SWEEP = True` to additionally reproduce the seed-variance
check for the three tiers (not required for any reported number, but useful
for judging how stable the tier comparison is).
